# Generate StrongREJECT scores for the six Qwen2.5 sizes

This notebook produces `results/scores.csv`, the input to `analysis.ipynb`. For every
harmful behavior it runs four arms against each Qwen2.5-Instruct size and grades the reply
with the StrongREJECT rubric judge:

- `none` - the behavior as written (baseline for lift)
- `dan` - a fixed in-the-wild DAN template (Shen et al.), selected by ID
- `pap` - persuasion rewrite from the `strong_reject` package
- `crescendo` - PyRIT's multi-turn Crescendo (or PAIR fallback; see config)

**Attack text is never authored here.** DAN, PAP, Crescendo, and PAIR all come from published
research code. This notebook only wires them to the target models and the judge.

**Safety / terms.** This assumes the academic-review approval the user stated. Running these
published attacks against models you host on Together for safety evaluation is consistent
with Together's documented red-teaming use; the judge call is ordinary OpenAI API usage.
Confirm both providers' current terms before a full run. Raw prompts and replies are written
under `results/raw/` (git-ignored); only `scores.csv` and figures are shared.

**Cost.** The six sizes run on Together *dedicated* endpoints that bill by the hour (serverless
has no Qwen2.5). Deploy each endpoint out of band, set its model string in the environment
(see the config cell), run smallest-first, and tear endpoints down afterwards.

In [1]:
import os, sys, json, time, platform
import jbscale
from jbscale import config

# Required credentials. Targets + attacker use Together; the judge uses OpenAI;
# the gated DAN dataset needs a Hugging Face token.
need = [config.TOGETHER_API_KEY_ENV, config.OPENAI_API_KEY_ENV, config.HF_TOKEN_ENV]
missing = [k for k in need if not os.environ.get(k)]
print('Python:', sys.version.split()[0], '|', platform.platform())
for k in need:
    print(f'  {k}: ' + ('set' if os.environ.get(k) else 'MISSING'))
if missing:
    print('\nSet the missing variables before running the smoke test or full run.')
HAVE_KEYS = not missing

Python: 3.12.1 | Windows-11-10.0.26300-SP0
  TOGETHER_API_KEY: MISSING
  OPENAI_API_KEY: MISSING
  HF_TOKEN: MISSING

Set the missing variables before running the smoke test or full run.


## Configuration

Everything that defines the run lives in `jbscale/config.py`. The key thing to set per
environment is the deployed Together model string for each size, via `JBSCALE_TARGET_<KEY>`
environment variables (e.g. `JBSCALE_TARGET_32B`) or `TARGET_ENDPOINT_OVERRIDES`. The 32B
model is deployable from Together's dedicated catalog; the smaller sizes are uploaded once
from their Hugging Face repos (see `jbscale/config.py` for the repo IDs).

In [2]:
for m in config.TARGET_MODELS:
    print(f'{m.key:>5}  n_params={m.n_params:>14,}  -> {config.target_model_string(m)}')
print('\nAttacker model:', config.ATTACKER_MODEL)
print('Judge:', config.JUDGE_EVALUATOR, config.JUDGE_MODELS)
print('PAP techniques:', config.PAP_TECHNIQUES)
print('Adaptive arm:', config.ADAPTIVE_ARM,
      '(Crescendo max_turns=%d, max_backtracks=%d)' % (config.CRESCENDO_MAX_TURNS, config.CRESCENDO_MAX_BACKTRACKS))

 0.5b  n_params=   494,032,768  -> Qwen/Qwen2.5-0.5B-Instruct
 1.5b  n_params= 1,543,714,304  -> Qwen/Qwen2.5-1.5B-Instruct
   3b  n_params= 3,085,938,688  -> Qwen/Qwen2.5-3B-Instruct
   7b  n_params= 7,615,616,512  -> Qwen/Qwen2.5-7B-Instruct
  14b  n_params=14,770,033,664  -> Qwen/Qwen2.5-14B-Instruct
  32b  n_params=32,763,876,352  -> Qwen/Qwen2.5-32B-Instruct

Attacker model: openai/gpt-oss-120b
Judge: strongreject_rubric ('openai/gpt-4o-mini',)
PAP techniques: ('logical_appeal',)
Adaptive arm: crescendo (Crescendo max_turns=5, max_backtracks=2)


## Smoke test

Before the expensive full run, exercise the whole path on a few behaviors and the smallest
model only: load data, build each attack, query the target, and grade. This confirms the
endpoint responds and the judge returns scores, and prints a rough cost estimate. It needs
the 0.5B endpoint deployed and the keys set.

In [3]:
if HAVE_KEYS:
    from jbscale.data import load_behaviors
    from jbscale.runner import prepare_pap_rewrites, run_model
    from jbscale.store import ResultStore

    smoke_target = next(m for m in config.TARGET_MODELS if m.key == config.SMOKE_TARGET_KEY)
    harmful = load_behaviors('harmful', limit=config.SMOKE_N_BEHAVIORS)
    benign = load_behaviors('benign', limit=config.SMOKE_N_BEHAVIORS)
    print(f'Smoke: {len(harmful)} harmful + {len(benign)} benign on {smoke_target.key}')

    smoke_store = ResultStore(os.path.join(config.PATHS.raw, 'smoke_records.jsonl'))
    pap_cache = prepare_pap_rewrites(harmful,
                                     store_path=os.path.join(config.PATHS.raw, 'smoke_pap.jsonl'))
    t0 = time.time()
    run_model(smoke_target, harmful, benign, pap_cache, smoke_store)
    print('Smoke run finished in %.1fs' % (time.time() - t0))
    import pandas as pd
    df = pd.DataFrame(smoke_store.records())
    display(df.groupby('attack')['strongreject'].mean())
else:
    print('Keys missing: skipping smoke test.')

Keys missing: skipping smoke test.


In [4]:
# Rough cost estimate for the full run (very approximate; refine with smoke timings).
n_sizes = len(config.TARGET_MODELS)
n_harmful = config.N_BEHAVIORS
arms = len(config.ATTACKS)
target_calls = n_sizes * n_harmful * arms + n_sizes * n_harmful  # +benign baseline
print('Target completions (excluding multi-turn/attacker overhead):', target_calls)
print('Judge calls (harmful arms + benign):', n_sizes * n_harmful * (arms) + n_sizes * n_harmful)
print('Note: Crescendo/PAIR add several attacker+target calls per behavior.')
print('Note: dedicated endpoints also bill per hour of uptime, not just per token.')

Target completions (excluding multi-turn/attacker overhead): 3000
Judge calls (harmful arms + benign): 3000
Note: Crescendo/PAIR add several attacker+target calls per behavior.
Note: dedicated endpoints also bill per hour of uptime, not just per token.


## Prepare PAP rewrites once

Persuasion rewrites are target-independent, so they are generated a single time with the
fixed attacker model and reused at every size. Rewrites are cached in `results/raw/` and the
attacker's refusal rate is reported.

In [5]:
if HAVE_KEYS:
    from jbscale.data import load_behaviors
    from jbscale.runner import prepare_pap_rewrites, attacker_refusal_rate
    harmful_full = load_behaviors('harmful', limit=config.N_BEHAVIORS)
    pap_cache_full = prepare_pap_rewrites(harmful_full)
    print('PAP rewrites cached:', len(pap_cache_full))
    print('Attacker refusal rate: %.3f' % attacker_refusal_rate(pap_cache_full))
else:
    print('Keys missing: skipping PAP preparation.')

Keys missing: skipping PAP preparation.


## Full run across the six sizes

Runs every arm for all 100 harmful behaviors plus the benign baseline, smallest size first.
The run is resumable: finished (behavior, size, attack) cells are skipped, so you can deploy
one endpoint, run its size, tear it down, and move to the next. **This spends money on
Together and OpenAI** and is intended to be started deliberately by the team, not re-run on
every notebook execution.

In [6]:
RUN_FULL = False  # set True once endpoints are deployed and you intend to spend

if RUN_FULL and HAVE_KEYS:
    from jbscale.data import load_behaviors
    from jbscale.runner import run_model
    from jbscale.store import ResultStore
    store = ResultStore()
    benign_full = load_behaviors('benign', limit=config.N_BEHAVIORS)
    for target in config.TARGET_MODELS:  # already smallest-first
        print('=== running', target.key, '===')
        t0 = time.time()
        run_model(target, harmful_full, benign_full, pap_cache_full, store)
        print('  done in %.1fs' % (time.time() - t0))
    print('All sizes complete.')
else:
    print('RUN_FULL is False (or keys missing). Not running the paid full sweep.')

RUN_FULL is False (or keys missing). Not running the paid full sweep.


## Export scores.csv and a run manifest

Flatten the stored records into the long `results/scores.csv` the analysis notebook reads, and
write a manifest recording package versions, the attacker model, and the attacker refusal rate
for reproducibility.

In [7]:
if RUN_FULL and HAVE_KEYS:
    path = store.export_scores()
    import importlib.metadata as im
    def ver(p):
        try:
            return im.version(p)
        except Exception:
            return 'unknown'
    manifest = {
        'attacker_model': config.ATTACKER_MODEL,
        'judge_models': list(config.JUDGE_MODELS),
        'adaptive_arm': config.ADAPTIVE_ARM,
        'pap_techniques': list(config.PAP_TECHNIQUES),
        'attacker_refusal_rate': attacker_refusal_rate(pap_cache_full),
        'target_model_strings': {m.key: config.target_model_string(m) for m in config.TARGET_MODELS},
        'versions': {p: ver(p) for p in ['strong_reject', 'pyrit', 'openai', 'datasets']},
    }
    with open(config.PATHS.manifest, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
    print('Wrote', path)
    print('Wrote', config.PATHS.manifest)
else:
    print('Nothing to export (full run not executed this session).')

Nothing to export (full run not executed this session).
